# Adquisición reproducible de las fuentes del seguimiento distrital

## 1. Propósito y delimitación

En este notebook localizo y descargo las fuentes oficiales del plan configurado. Mi alcance comprende adquisición, exploración y limpieza reproducible. Trabajo con Avance del Plan y Gestión-Inversión; cuando activo narrativas necesito además el PDF por entidades. Conservo los bytes originales en Bronce y preparo sus datos en los notebooks siguientes.

Selecciono explícitamente el plan y su año inicial. No equiparo observaciones de planes diferentes por compartir una fecha o parte de sus códigos. Distingo corte de publicación, fecha del dato y versión del archivo. El año final de búsqueda procede del catálogo, no de una fecha escrita en el documento.

**Fuente primaria:** [catálogo de Programación y seguimiento a la inversión de la SDP](https://www.sdp.gov.co/gestion-a-la-inversion/programacion-y-seguimiento-a-la-inversion/seguimiento).

**Lectura de los resultados.** La metodología describe el procedimiento para cualquier corte compatible. Los resúmenes que aparecen al ejecutar las celdas describen exclusivamente la fuente seleccionada. Para actualizar el estudio ejecuto el notebook desde el inicio; las salidas guardadas no se actualizan por abrir el archivo. Mantengo las revisiones anteriores en un anexo histórico, sin atribuirlas al corte actual.

La instalación siguiente solo hace falta si todavía no dispongo de las dependencias.



In [ ]:
# Si faltan las dependencias, descomenta y ejecuta:
# %pip install "requests>=2.32,<3" "beautifulsoup4>=4.12,<5" "pypdf>=5,<7"

## 2. Configuración del alcance y del corte

Declaro el plan, su sigla y el año inicial como parámetros de alcance. Con `CORTE_DESCARGA=None` busco el último corte compatible; si escribo una fecha ISO exijo ese periodo y no lo sustituyo silenciosamente.

Por defecto, `INCLUIR_NARRATIVAS=False` requiere los Excel de ambas fuentes. Con `True` exijo además el PDF de Gestión-Inversión por entidades. No hago depender la limpieza numérica de una infografía ni de un PDF opcional. La disponibilidad efectiva y el corte seleccionado se muestran al ejecutar la consulta; no los fijo en esta explicación.

La referencia histórica del notebook 03 sirve para contrastar mediciones; no constituye una descarga operativa de todos los cortes ni determina cuál es el último disponible.



In [ ]:
# Lectura temporal del PDF para identificar el tipo de reporte.
from io import BytesIO
from pypdf import PdfReader
from dataclasses import dataclass
from datetime import date, datetime
from pathlib import Path
from urllib.parse import urljoin, urlparse, unquote
from zoneinfo import ZoneInfo
import calendar
import hashlib
import re
import time
import unicodedata
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Parámetros editables: portal, plan y límite inferior de búsqueda.
PORTAL_URL = "https://www.sdp.gov.co/gestion-a-la-inversion/programacion-y-seguimiento-a-la-inversion/seguimiento"
PLAN_NOMBRE = "Bogotá Camina Segura"
PLAN_SIGLA = "BCS"
ANIO_INICIAL = 2024
CORTE_DESCARGA = None  # None: último compatible; o una fecha ISO exacta.
INCLUIR_NARRATIVAS = False  # Último corte numérico; True exige también PDF por entidades.

# Carpeta donde se guardan los archivos originales descargados.
# Ejecutar desde la raíz del proyecto; no se fija una ruta personal absoluta.
# Buscar la carpeta del proyecto, también al abrir un notebook desde una subcarpeta.
import os
import sys
sys.dont_write_bytecode = True
_carpeta_indicada = os.environ.get("PROYECTO_RAIZ")
_candidatas = ([Path(_carpeta_indicada).expanduser().resolve()] if _carpeta_indicada
               else [Path.cwd().resolve(), *Path.cwd().resolve().parents])
RAIZ = next((p for p in _candidatas if (p / "rutas.py").is_file()
             and (p / "01_descarga_ultimo_corte.ipynb").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("Abre Jupyter desde la carpeta del proyecto.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from rutas import raiz_proyecto
RAIZ = raiz_proyecto()
DESTINO = RAIZ / "Capa Bronce"


## 3. Normalización de etiquetas del catálogo

Normalizo espacios, acentos y mayúsculas únicamente para comparar etiquetas del portal. No aplico esta función al contenido de los archivos descargados. Convierto las etiquetas de mes en fechas de cierre para ordenar los periodos, en vez de fijar el último trimestre en el código.

Represento opciones, formularios, publicaciones y adjuntos mediante estructuras explícitas. Así puedo distinguir el periodo consultado, la dirección del detalle y el enlace al archivo, sin tratar el nombre de un archivo como si fuera por sí solo evidencia de su contenido.


In [ ]:
# Normaliza etiquetas para compararlas; no modifica el contenido de los archivos.
def normalize_text(value):
    text = unicodedata.normalize("NFKD", str(value or ""))
    text = "".join(c for c in text if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", text).strip().lower()

MONTHS = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4,
          "mayo": 5, "junio": 6, "julio": 7, "agosto": 8,
          "septiembre": 9, "setiembre": 9, "octubre": 10,
          "noviembre": 11, "diciembre": 12}

# Convierte la etiqueta del mes en su último día calendario para ordenar los cortes.
def fecha_corte(year, label):
    month = MONTHS[normalize_text(label).split()[-1]]
    return date(year, month, calendar.monthrange(year, month)[1])

FILTERS = {
    "clasificacion": {
        "id": "edit-field-inversion-segui-class-tid",
        "name": "field_inversion_segui_class_tid",
    },
    "periodo": {
        "id": "edit-field-inversion-segui-periodo-tid",
        "name": "field_inversion_segui_periodo_tid",
    },
    "vigencia": {
        "id": "edit-field-inversion-segui-vigencia-value-value-year",
        "name": "field_inversion_segui_vigencia_value[value][year]",
    },
}

# Representa una opción visible y su valor en el formulario del portal.
@dataclass(frozen=True)
class Option:
    label: str
    value: str

# Conserva la acción, el método y los campos ocultos del formulario.
@dataclass(frozen=True)
class FormSpec:
    action_url: str
    method: str
    defaults: dict[str, str]

# Representa una publicación del catálogo y el enlace a su detalle.
@dataclass(frozen=True)
class Publication:
    title: str
    detail_url: str

# Representa un adjunto original y sus datos publicados en el portal.
@dataclass(frozen=True)
class Attachment:
    title: str
    publication_date: str
    url: str



## 4. Conexión y control de errores de adquisición

Centralizo solicitudes, tiempos de espera y reintentos de errores transitorios. Mantengo la verificación de certificados. Distingo falta de un formato publicado de un fallo de conexión o de lectura: una consulta fallida no demuestra que la fuente no exista.



In [ ]:
# Centraliza las solicitudes HTTP, los reintentos y las pausas entre consultas.
class SDPClient:
    """Sesión HTTP con reintentos y pausa responsable entre solicitudes."""

    # Configura conexión, tiempos de espera y tratamiento de errores temporales.
    def __init__(self, config: dict) -> None:
        portal = config["portal"]
        self.base_url = portal["url"]
        self.timeout = (
            int(portal.get("connect_timeout", 15)),
            int(portal.get("read_timeout", 120)),
        )
        self.delay = float(portal.get("delay_seconds", 0.8))
        self.verify_ssl = bool(portal.get("verify_ssl", True))
        self.last_request = 0.0

        retry = Retry(
            total=int(portal.get("retries", 3)),
            connect=int(portal.get("retries", 3)),
            read=int(portal.get("retries", 3)),
            status=int(portal.get("retries", 3)),
            backoff_factor=float(portal.get("backoff", 1.0)),
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=frozenset({"GET", "HEAD", "POST"}),
            raise_on_status=False,
        )
        adapter = HTTPAdapter(max_retries=retry)
        self.session = requests.Session()
        self.session.mount("http://", adapter)
        self.session.mount("https://", adapter)
        self.session.headers.update({
            "User-Agent": "Mozilla/5.0 (compatible; SDP-BCS-Pipeline/1.0)",
            "Accept-Language": "es-CO,es;q=0.9",
        })

    # Permite abrir la sesión mediante un bloque with.
    def __enter__(self) -> "SDPClient":
        return self

    # Cierra la sesión al terminar, incluso si ocurre un error.
    def __exit__(self, exc_type, exc, tb) -> None:
        self.session.close()

    # Respeta la pausa entre solicitudes y comunica los errores HTTP.
    def _request(self, method: str, url: str, **kwargs) -> requests.Response:
        wait = self.delay - (time.monotonic() - self.last_request)
        if wait > 0:
            time.sleep(wait)
        response = self.session.request(
            method,
            url,
            timeout=kwargs.pop("timeout", self.timeout),
            verify=kwargs.pop("verify", self.verify_ssl),
            **kwargs,
        )
        self.last_request = time.monotonic()
        response.raise_for_status()
        return response

    # Obtiene páginas o archivos del portal.
    def get(self, url: str, **kwargs) -> requests.Response:
        return self._request("GET", url, **kwargs)

    # Consulta los encabezados HTTP para reconocer formatos sin leer tablas.
    def head(self, url: str) -> requests.Response:
        return self._request("HEAD", url, allow_redirects=True)

    # Envía los filtros conservando los campos ocultos del formulario.
    def submit(self, form: FormSpec, fields: dict[str, str]) -> requests.Response:
        payload = dict(form.defaults)
        payload.update(fields)
        if form.method == "POST":
            return self._request("POST", form.action_url, data=payload)
        return self._request("GET", form.action_url, params=payload)



## 5. Lectura de filtros, publicaciones y adjuntos

Leo los años, periodos y clasificaciones ofrecidos por el formulario y recorro la paginación de sus resultados. Después abro el detalle de cada publicación y recupero sus enlaces. Con este procedimiento puedo incorporar nuevos trimestres sin añadir manualmente una URL por periodo.

Identifico tablas por sus encabezados y verifico la presencia de los filtros que necesito. Si cambia el portal y no puedo reconocer su estructura, detengo la adquisición. No interpreto una página inesperada como una lista vacía de fuentes.


In [ ]:
# Interpreta el HTML recibido para localizar formularios y enlaces.
def _soup(html: str | bytes) -> BeautifulSoup:
    return BeautifulSoup(html, "html.parser")

# Lee los años, periodos y clasificaciones ofrecidos por la SDP.
def _discover_options(html: str | bytes) -> dict[str, list[Option]]:
    soup = _soup(html)
    result: dict[str, list[Option]] = {}
    for key, metadata in FILTERS.items():
        select = soup.find("select", id=metadata["id"]) or soup.find(
            "select", attrs={"name": metadata["name"]}
        )
        if select is None:
            raise RuntimeError(f"El portal cambió: no se encontró el filtro {key!r}.")
        result[key] = [
            Option(option.get_text(" ", strip=True), str(option.get("value", "")).strip())
            for option in select.find_all("option")
            if str(option.get("value", "")).strip()
        ]
    return result

# Detecta cómo se envían los filtros; no fija manualmente la URL de resultados.
def _discover_form(html: str | bytes, base_url: str) -> FormSpec:
    soup = _soup(html)
    select = soup.find("select", id=FILTERS["clasificacion"]["id"])
    if select is None:
        raise RuntimeError("No se encontró el formulario de filtros de la SDP.")
    form = select.find_parent("form")
    if form is None:
        return FormSpec(base_url, "GET", {})

    method = str(form.get("method", "GET")).upper()
    action = urljoin(base_url, form.get("action") or base_url)
    defaults: dict[str, str] = {}
    for field in form.find_all("input"):
        if str(field.get("type", "text")).lower() != "hidden":
            continue
        name = field.get("name")
        if name:
            defaults[str(name)] = str(field.get("value", ""))
    return FormSpec(action, method, defaults)

# Homologa encabezados HTML para reconocer las tablas del catálogo.
def _table_headers(table) -> list[str]:
    return [normalize_text(cell.get_text(" ", strip=True)) for cell in table.find_all("th")]

# Busca una tabla HTML por sus encabezados, no por su posición en la página.
def _find_table(soup: BeautifulSoup, required: set[str]):
    required_norm = {normalize_text(value) for value in required}
    for table in soup.find_all("table"):
        if required_norm.issubset(set(_table_headers(table))):
            return table
    return None

# Recupera títulos y enlaces de las publicaciones del resultado filtrado.
def _extract_publications(html: str | bytes, page_url: str) -> list[Publication]:
    soup = _soup(html)
    table = _find_table(soup, {"Nombre", "Clasificación", "Periodo", "Vigencia", "Detalle"})
    if table is None:
        # En combinaciones sin resultados Drupal puede devolver solo el formulario.
        return []

    headers = _table_headers(table)
    detail_idx = headers.index("detalle") if "detalle" in headers else -1
    title_idx = headers.index("nombre") if "nombre" in headers else -1
    tbody = table.find("tbody")
    trs = tbody.find_all("tr") if tbody else table.find_all("tr")[1:]

    result: list[Publication] = []
    for tr in trs:
        tds = tr.find_all("td")
        if not tds:
            continue
        link = tds[detail_idx].find("a", href=True) if 0 <= detail_idx < len(tds) else tr.find("a", href=True)
        if link is None:
            continue
        title = tds[title_idx].get_text(" ", strip=True) if 0 <= title_idx < len(tds) else ""
        result.append(Publication(title, urljoin(page_url, link["href"])))
    return result

# Localiza la siguiente página de resultados, cuando existe paginación.
def _next_page(html: str | bytes, page_url: str) -> str | None:
    soup = _soup(html)
    for selector in ["li.pager-next a[href]", ".pager-next a[href]", 'a[rel="next"][href]']:
        link = soup.select_one(selector)
        if link is not None:
            return urljoin(page_url, link["href"])
    return None

# Lee los enlaces originales de descarga dentro de cada publicación.
def _extract_attachments(html: str | bytes, detail_url: str) -> list[Attachment]:
    soup = _soup(html)
    table = _find_table(soup, {"Titulo", "Descripción", "Fecha de Publicación", "Enlace"})
    if table is None:
        return []

    headers = _table_headers(table)
    link_idx = headers.index("enlace") if "enlace" in headers else -1
    title_idx = headers.index("titulo") if "titulo" in headers else -1
    date_idx = headers.index("fecha de publicacion") if "fecha de publicacion" in headers else -1
    tbody = table.find("tbody")
    trs = tbody.find_all("tr") if tbody else table.find_all("tr")[1:]

    result: list[Attachment] = []
    for tr in trs:
        tds = tr.find_all("td")
        if not tds:
            continue
        link = tds[link_idx].find("a", href=True) if 0 <= link_idx < len(tds) else tr.find("a", href=True)
        if link is None:
            continue
        result.append(Attachment(
            title=tds[title_idx].get_text(" ", strip=True) if 0 <= title_idx < len(tds) else "",
            publication_date=tds[date_idx].get_text(" ", strip=True) if 0 <= date_idx < len(tds) else "",
            url=urljoin(detail_url, link["href"]),
        ))
    return result

# Localiza una opción por su etiqueta normalizada.
def _find_option(options: list[Option], label: str) -> Option | None:
    target = normalize_text(label)
    return next((option for option in options if normalize_text(option.label) == target), None)

# Incluye cortes mensuales regulares y excluye etiquetas extraordinarias.
def _regular_cuts(options: list[Option]) -> list[Option]:
    """Acepta cualquier futuro `Corte <mes>` y excluye cortes extraordinarios."""
    selected: list[Option] = []
    for option in options:
        label = normalize_text(option.label)
        if re.fullmatch(r"corte\s+(enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|setiembre|octubre|noviembre|diciembre)", label):
            selected.append(option)
    return selected

# Consulta una combinación de fuente, año y corte, recorriendo la paginación.
def _query_publications(
    client: SDPClient,
    form: FormSpec,
    class_option: Option,
    year_option: Option,
    cut_option: Option,
) -> list[Publication]:
    """Consulta una combinación clasificación/año/corte incluyendo paginación."""
    fields = {
        FILTERS["clasificacion"]["name"]: class_option.value,
        FILTERS["periodo"]["name"]: cut_option.value,
        FILTERS["vigencia"]["name"]: year_option.value,
    }
    response = client.submit(form, fields)
    page_html, page_url = response.text, response.url
    publications = _extract_publications(page_html, page_url)

    while True:
        next_url = _next_page(page_html, page_url)
        if not next_url:
            break
        page = client.get(next_url)
        page_html, page_url = page.text, page.url
        publications.extend(_extract_publications(page_html, page_url))

    return list({item.detail_url: item for item in publications}.values())

## 6. Identificación de formatos y selección del plan

Reconozco Excel y PDF por extensión y, cuando hace falta, por encabezados HTTP. Filtro las publicaciones por el nombre o la sigla configurados y mantengo separadas Avance del Plan y Gestión-Inversión. No incorporo automáticamente otra clasificación porque cambiaría la cobertura del estudio.



In [ ]:
# Reconoce Excel o PDF por extensión y, si hace falta, por encabezados HTTP.
def formato_adjunto(client, adjunto):
    extension = Path(urlparse(adjunto.url).path).suffix.lower()
    if extension in {".xlsx", ".xls"}:
        return "Excel"
    if extension == ".pdf":
        return "PDF"
    with client.head(adjunto.url) as respuesta:
        tipo = respuesta.headers.get("Content-Type", "").lower()
        nombre = respuesta.headers.get("Content-Disposition", "").lower()
    if "application/pdf" in tipo or ".pdf" in nombre:
        return "PDF"
    if "spreadsheetml" in tipo or "ms-excel" in tipo or ".xlsx" in nombre or ".xls" in nombre:
        return "Excel"
    return None


FUENTES = {
    "avance_plan": "Avance del Plan de Desarrollo",
    "gestion_inversion": "Gestión-Inversión",
}

# Filtra títulos de publicaciones por el nombre o la sigla del PDD configurado.
def corresponde_al_plan(titulo):
    titulo = normalize_text(titulo)
    nombre = normalize_text(PLAN_NOMBRE)
    sigla = normalize_text(PLAN_SIGLA)
    return nombre in titulo or bool(re.search(r"\b" + re.escape(sigla) + r"\b", titulo))

# Permite comparar nombres de clasificación con diferencias de espacios o signos.
def clave_clasificacion(text):
    return re.sub(r"[^a-z0-9]", "", normalize_text(text))



## 7. Reconocimiento del PDF por entidades

Inspecciono temporalmente las primeras páginas de los PDF candidatos para confirmar plan, corte, organización por entidades y rótulos narrativos. Admito las variantes de nombre contempladas por el reconocedor; el nombre del archivo no sustituye la identificación por contenido.

Descarto vistas por sector o estructura y agrupo copias binarias idénticas mediante SHA-256. Si encuentro versiones diferentes del reporte requerido, detengo la selección para resolver la fuente. La extracción de los relatos corresponde al notebook 04.



In [ ]:
# Identifica el reporte por su contenido; no confía en el nombre del archivo.
def clasificar_pdf_gestion(contenido, corte):
    """Lee encabezados del PDF, sin transformar sus datos ni depender del nombre."""
    try:
        reader = PdfReader(BytesIO(contenido))
        if reader.is_encrypted:
            raise ValueError("PDF cifrado")
        paginas = [page.extract_text() or "" for page in reader.pages[:2]]
    except Exception as exc:
        raise RuntimeError("No se pudo leer el PDF para identificar su reporte.") from exc
    if not paginas or not any(t.strip() for t in paginas):
        raise RuntimeError("PDF sin texto legible; requiere revision u OCR antes de seleccionar.")
    cabecera = normalize_text(paginas[0][:1800])
    if normalize_text(PLAN_NOMBRE) not in cabecera:
        raise RuntimeError("El encabezado del PDF no confirma el PDD configurado.")
    fechas = re.findall(r"fecha\s+de\s+corte\s*:\s*(\d{1,2})/(\d{1,2})/(\d{4})", cabecera)
    if not fechas or any(date(int(y), int(m), int(d)) != corte for d, m, y in fechas):
        raise RuntimeError("El PDF no confirma la misma fecha de corte que la publicacion.")
    if "reporte" not in cabecera or "gestion" not in cabecera or "inversion" not in cabecera:
        raise RuntimeError("El encabezado no identifica un reporte de gestion e inversion.")
    if re.search(r"por\s+(?:por\s+)?(?:sector|estructura)", cabecera):
        return "otra_vista"
    por_entidad = bool(re.search(r"por\s+(?:entidad(?:es)?|alcaldia(?:s)?)", cabecera))
    # Permite un titulo diferente si la organizacion por entidad sigue visible.
    entidad_cabecera = bool(re.search(r"\b\d{4}\s*-\s*[a-z]", cabecera))
    jerarquia = all(palabra in cabecera for palabra in ("indicador", "proyecto", "actividad"))
    relatos = normalize_text(" ".join(paginas))
    if (por_entidad or (entidad_cabecera and jerarquia)) and "avances y logros" in relatos and "retrasos y soluciones" in relatos:
        return "entidades"
    raise RuntimeError("PDF no reconocido: no se puede confirmar el reporte por entidades. No se elige un corte anterior silenciosamente.")


# Elige un único reporte por entidades y descarta otras vistas sin guardarlas.
def seleccionar_pdf_gestion(client, adjuntos, corte):
    candidatos = {}
    for adjunto in adjuntos:
        # Revalido la fuente remota: una misma URL puede publicar una revisión.
        with client.get(adjunto.url) as respuesta:
            contenido = respuesta.content
        tipo = clasificar_pdf_gestion(contenido, corte)
        print(f"  PDF identificado por contenido: {tipo}", flush=True)
        if tipo == "entidades":
            huella = hashlib.sha256(contenido).hexdigest()
            candidatos.setdefault(huella, (adjunto, contenido))
    if len(candidatos) > 1:
        raise RuntimeError("Existen varios reportes por entidades con contenido diferente; se requiere elegir la version.")
    if adjuntos and not candidatos:
        raise RuntimeError("Hay PDF publicados pero ninguno es el reporte por entidades. Se detiene la seleccion.")
    if not candidatos:
        return []
    adjunto, contenido = next(iter(candidatos.values()))
    # Reutilizar los bytes inspeccionados evita descargar nuevamente el elegido.
    client.pdf_seleccionados = {adjunto.url: contenido}
    return [adjunto]




## 8. Selección del último corte compatible

Consulto años y cortes en orden descendente y compruebo los insumos del modo elegido. No admito fechas futuras ni considero completo un periodo por la sola presencia de una publicación en el catálogo.

Distingo compatibilidad numérica de disponibilidad narrativa. Si falta un insumo de un corte solicitado expresamente, comunico el faltante; si hay versiones distintas, no elijo una por su posición en la página. La limpieza utiliza un corte local común para evitar combinar periodos de ambas fuentes.



In [ ]:
# Busca hacia atrás una misma fecha con todos los archivos exigidos.
def buscar_ultimo_corte_completo(client, form, options):
    """Ultimo corte regular con Excel y PDF para TODAS las fuentes."""
    clases = {}
    for fuente, etiqueta in FUENTES.items():
        opcion = next((o for o in options["clasificacion"]
                       if clave_clasificacion(o.label) == clave_clasificacion(etiqueta)), None)
        if opcion is None:
            raise RuntimeError(f"Clasificacion no encontrada: {etiqueta}")
        clases[fuente] = opcion
    years = sorted((o for o in options["vigencia"]
                    if o.label.isdigit() and int(o.label) >= ANIO_INICIAL),
                   key=lambda o: int(o.label), reverse=True)
    cuts = _regular_cuts(options["periodo"])
    if not years or not cuts:
        raise RuntimeError("No se encontraron anios o cortes regulares.")
    for year in years:
        for cut in sorted(cuts, key=lambda o: fecha_corte(int(year.label), o.label), reverse=True):
            fecha = fecha_corte(int(year.label), cut.label)
            if CORTE_DESCARGA and fecha.isoformat() != CORTE_DESCARGA:
                continue
            if fecha > datetime.now(ZoneInfo("America/Bogota")).date():
                continue
            print(f"Consultando corte comun: {fecha:%d/%m/%Y}", flush=True)
            seleccion = {}
            for fuente, clase in clases.items():
                publicaciones = _query_publications(client, form, clase, year, cut)
                adjuntos = {}
                for publicacion in publicaciones:
                    if not corresponde_al_plan(publicacion.title):
                        continue
                    respuesta = client.get(publicacion.detail_url)
                    for adjunto in _extract_attachments(respuesta.content, respuesta.url):
                        adjuntos[adjunto.url] = adjunto
                formatos = {"Excel": [], "PDF": []}
                for adjunto in adjuntos.values():
                    formato = formato_adjunto(client, adjunto)
                    if formato in formatos:
                        formatos[formato].append(adjunto)
                if fuente == "gestion_inversion" and INCLUIR_NARRATIVAS:
                    formatos["PDF"] = seleccionar_pdf_gestion(client, formatos["PDF"], fecha)
                if len(formatos["Excel"]) > 1:
                    raise RuntimeError(f"Hay varios Excel para {fuente}/{fecha}; se requiere resolver la versión.")
                if not INCLUIR_NARRATIVAS:
                    formatos["PDF"] = []
                completo = bool(formatos["Excel"]) and (fuente != "gestion_inversion" or not INCLUIR_NARRATIVAS or bool(formatos["PDF"]))
                if not completo:
                    print(f"  {FUENTES[fuente]}: faltan insumos requeridos; reviso el siguiente corte elegible.", flush=True)
                    break
                seleccion[fuente] = formatos
            if CORTE_DESCARGA and len(seleccion) != len(FUENTES):
                raise RuntimeError(f"El corte solicitado {fecha} no tiene los insumos requeridos; no lo sustituyo por otro.")
            if len(seleccion) == len(FUENTES):
                return fecha, seleccion
    raise RuntimeError("No existe un corte regular comun con Excel y PDF para ambas fuentes en el rango configurado.")




## 9. Persistencia e integridad de las fuentes

Guardo los bytes originales mediante una escritura temporal y una sustitución al finalizar. El sufijo derivado de la URL distingue enlaces con nombres iguales; la huella SHA-256 identifica el contenido, y no confundo ambos identificadores.

Revalido el contenido remoto incluso cuando ya existe un archivo local. Si sus bytes son iguales, reutilizo la copia. Si la misma URL cambió de contenido, conservo el original y detengo el flujo para resolver la revisión: no sobrescribo evidencia ni supongo que una versión nueva sea equivalente a la anterior. Tampoco elimino cortes históricos.

Los controles permiten repetir una ejecución y detectar sustituciones del proveedor. Una interrupción de la conexión puede requerir reejecutar; no doy por válida una descarga parcial.


In [ ]:
# Calcula SHA-256 por bloques para reconocer archivos idénticos byte a byte.
def hash_archivo(ruta):
    resumen = hashlib.sha256()
    with ruta.open("rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            resumen.update(bloque)
    return resumen.hexdigest()


# Guarda el original, reutiliza descargas existentes y evita copias idénticas.
def descargar_original(client, adjunto, carpeta):
    identificador = hashlib.sha256(adjunto.url.encode()).hexdigest()[:10]
    # Una URL ya guardada no se descarga de nuevo.
    existentes = [p for p in carpeta.glob(f"*_{identificador}.*")
                  if p.is_file() and p.suffix.lower() in {".xlsx", ".xls", ".pdf"}
                  and p.stat().st_size > 0]
    contenido_previo = getattr(client, "pdf_seleccionados", {}).get(adjunto.url)
    if contenido_previo is not None:
        carpeta_pdf = carpeta
        carpeta_pdf.mkdir(parents=True, exist_ok=True)
        huella = hashlib.sha256(contenido_previo).hexdigest()
        if existentes and all(hash_archivo(p) != huella for p in existentes):
            raise RuntimeError(f"La URL del PDF cambió de contenido: {adjunto.url}. Conservo el original y detengo el procesamiento para resolver la revisión.")
        for existente in carpeta_pdf.glob("*.pdf"):
            if hash_archivo(existente) == huella:
                print(f"  PDF identico ya guardado: {existente.name}")
                return existente
        nombre = Path(unquote(urlparse(adjunto.url).path)).stem
        nombre = re.sub(r"[^a-zA-Z0-9_.-]", "_", nombre) or "reporte"
        ruta = carpeta_pdf / f"{nombre}_{identificador}.pdf"
        from tempfile import NamedTemporaryFile
        temporal = None
        try:
            with NamedTemporaryFile(dir=carpeta_pdf, suffix=".part", delete=False) as archivo:
                temporal = Path(archivo.name)
                archivo.write(contenido_previo)
            temporal.replace(ruta)
        finally:
            if temporal is not None:
                temporal.unlink(missing_ok=True)
        print(f"  Guardado sin repetir la descarga: {ruta.name}")
        return ruta
    # Los demas originales se descargan sin transformar su contenido.
    with client.get(adjunto.url, stream=True) as respuesta:
        tipo = respuesta.headers.get("Content-Type", "").lower()
        disposicion = unquote(respuesta.headers.get("Content-Disposition", ""))
        match = re.search(r"filename\*?\s*=\s*(?:UTF-8''|\")?([^\";]+)", disposicion, re.I)
        nombre = match.group(1).strip() if match else unquote(Path(urlparse(respuesta.url).path).name)
        nombre = re.sub(r'[<>:"/\\|?*\x00-\x1f]', '_', nombre).strip(' .') or 'archivo'
        extension = Path(nombre).suffix.lower()
        if extension not in {".xlsx", ".xls", ".pdf"}:
            extension = next((ext for mime, ext in [
                ("application/pdf", ".pdf"),
                ("application/vnd.openxmlformats-officedocument.spreadsheetml.sheet", ".xlsx"),
                ("application/vnd.ms-excel", ".xls") ] if mime in tipo), None)
            if extension is None:
                print(f"  Omitido: adjunto ajeno a Excel/PDF: {adjunto.title}")
                return None
            nombre += extension
        if "text/html" in tipo:
            raise RuntimeError(f"El servidor devolvió HTML en lugar del archivo: {adjunto.url}")
        # El sufijo estable evita colisiones entre enlaces con igual nombre.
        identificador = hashlib.sha256(adjunto.url.encode()).hexdigest()[:10]
        ruta = carpeta / (Path(nombre).stem + "_" + identificador + extension)
        ruta.parent.mkdir(parents=True, exist_ok=True)
        # Archivo temporal único y sustitución atómica: evita guardar descargas parciales.
        from tempfile import NamedTemporaryFile
        temporal = None
        try:
            with NamedTemporaryFile(dir=ruta.parent, suffix=".part", delete=False) as archivo:
                temporal = Path(archivo.name)
                for bloque in respuesta.iter_content(1024 * 1024):
                    if bloque:
                        archivo.write(bloque)
            if temporal.stat().st_size == 0:
                raise RuntimeError(f"Descarga vacía: {adjunto.url}")
            huella_remota = hash_archivo(temporal)
            if existentes:
                iguales = [p for p in existentes if hash_archivo(p) == huella_remota]
                if iguales:
                    print(f"Fuente remota revalidada sin cambios: {iguales[0].name}")
                    return iguales[0]
                raise RuntimeError(f"La URL cambió de contenido: {adjunto.url}. Conservo el original; es necesario resolver la versión antes de limpiar.")
            if extension == ".pdf":
                # Enlaces o nombres distintos pueden contener el mismo PDF.
                huella = hash_archivo(temporal)
                for existente in ruta.parent.glob("*.pdf"):
                    if (existente.stat().st_size == temporal.stat().st_size
                            and hash_archivo(existente) == huella):
                        print(f"  PDF identico omitido; se conserva: {existente.name}")
                        return existente
            temporal.replace(ruta)
        finally:
            if temporal is not None:
                temporal.unlink(missing_ok=True)
    print(f"  {ruta.name}")
    return ruta


## 10. Ejecución y lectura de la adquisición

Consulto el catálogo, selecciono el corte y guardo o revalido los adjuntos por fecha y fuente. El resumen calculado a continuación informa qué periodo y archivos obtuve en esta ejecución. No reutilizo como resultado actual la disponibilidad observada en una consulta anterior.

Después ejecuto 02 y 03 y, cuando exista un PDF compatible, 04. Si falla la adquisición, no interpreto la ausencia de un resumen final como una descarga completada.



In [ ]:
# Parámetros operativos de conexión; los tiempos están expresados en segundos.
config_http = {"portal": {"url": PORTAL_URL, "connect_timeout": 15,
                           "read_timeout": 120, "retries": 3,
                           "backoff": 1.0, "delay_seconds": 0.8,
                           "verify_ssl": True}}
# Resumen en memoria de las fuentes adquiridas en esta ejecución.
descargas = {}
with SDPClient(config_http) as client:
    # Leer el catálogo y sus opciones actuales, sin fijar manualmente el año final.
    pagina = client.get(PORTAL_URL)
    options = _discover_options(pagina.content)
    form = _discover_form(pagina.content, pagina.url)
    # Resolver una única fecha antes de guardar cualquiera de las fuentes.
    corte, seleccion = buscar_ultimo_corte_completo(client, form, options)
    print(f"Corte unico seleccionado: {corte:%d/%m/%Y}", flush=True)
    # Guardar primero por fecha de corte y luego por fuente, sin subcarpetas por formato.
    for fuente, formatos in seleccion.items():
        print(FUENTES[fuente], flush=True)
        descargas[fuente] = {"fecha_corte": corte, "archivos": []}
        for formato, adjuntos in formatos.items():
            for adjunto in adjuntos:
                ruta = descargar_original(client, adjunto, DESTINO / corte.isoformat() / fuente)
                if ruta is None:
                    raise RuntimeError(f"No se pudo descargar el adjunto: {adjunto.url}")
                descargas[fuente]["archivos"].append(ruta)
print(f"Descarga terminada. Todos los archivos corresponden al {corte:%d/%m/%Y}. Carpeta: {DESTINO}")


## 11. Organización y alcance de Bronce

Guardo los originales en `Capa Bronce/AAAA-MM-DD/avance_plan` y `Capa Bronce/AAAA-MM-DD/gestion_inversion`. La fecha de carpeta identifica el corte seleccionado, no el día de descarga. Los formatos opcionales solo aparecen si fueron requeridos y publicados.

Conservo nombres e identificadores de descarga. Un nuevo corte genera su ubicación sin borrar los anteriores. La disponibilidad del catálogo, la interpretación de las cifras y la veracidad de los reportes institucionales son cuestiones distintas: esta etapa comprueba la adquisición bajo los criterios descritos.



### Resumen calculado de las fuentes adquiridas

Calculo esta lectura con las variables de la ejecución, sin fijar conteos ni conclusiones de otro corte.


In [ ]:
from IPython.display import display, Markdown
from datetime import datetime
from zoneinfo import ZoneInfo
_momento_informe = datetime.now(ZoneInfo("America/Bogota")).isoformat(timespec="seconds")
_filas_fuentes = []
for _fuente_informe, _detalle_informe in descargas.items():
    _filas_fuentes.append(f"- **{_fuente_informe}**: corte {_detalle_informe['fecha_corte'].isoformat()}, {len(_detalle_informe['archivos'])} archivo(s) guardado(s) o revalidado(s).")
display(Markdown(f"**Consulta completada:** {_momento_informe}. Plan configurado: **{PLAN_NOMBRE} ({PLAN_SIGLA})**.\n\n" + "\n".join(_filas_fuentes) + "\n\nEstos resultados describen esta consulta; la disponibilidad puede cambiar en la próxima ejecución."))


---

<details>
<summary>Anexo histórico: evidencias y redacción de las revisiones anteriores — no describe el corte seleccionado</summary>

Conservo estos textos como registro de revisiones puntuales, realizadas sobre las fuentes y fechas que cada apartado identifica. Sus conteos, ejemplos, resultados y límites pertenecen a esas versiones. No son resultados recalculados ni condiciones satisfechas para una nueva ejecución. La metodología operativa y los informes del corte seleccionado están en el cuerpo principal. Las revisiones históricas no se amplían automáticamente.

# Adquisición reproducible de las fuentes del seguimiento distrital

## 1. Propósito y delimitación

En este notebook localizo y descargo las fuentes oficiales que necesito para preparar el seguimiento del Plan Distrital de Desarrollo. Mi alcance comprende adquisición, exploración y limpieza reproducible. En esta etapa no evalúo causalidad ni transformo una afirmación institucional en un resultado verificado externamente.

Trabajo con dos fuentes: Avance del Plan, para el seguimiento físico de sus niveles, y Gestión-Inversión, para actividades, magnitudes y recursos. Cuando incluyo narrativas necesito además el reporte por entidades. Conservo los bytes originales en Bronce y separo su adquisición de las transformaciones de los notebooks 02, 03 y 04.

Al revisar desde 2024 encontré una transición entre UNCSAB, código 16, y Bogotá Camina Segura, código 17. No equiparo metas sectoriales, metas producto y actividades entre planes solo porque compartan una fecha o parte de sus identificadores. Selecciono explícitamente el plan en la configuración y dejo que cada adaptador compruebe el esquema recibido.

**Periodo de la revisión.** Realicé la revisión histórica el 29 de octubre de 2026 con las versiones que el catálogo de la SDP ofrecía ese día. Distingo el corte de la publicación, la fecha de la observación y la versión del archivo. Las cifras históricas de este texto son resultados de esa revisión; las tablas ejecutables se recalculan para la fuente seleccionada. No atribuyo las conclusiones de un archivo a periodos futuros sin repetir los controles.

**Fuente primaria:** [catálogo de Programación y seguimiento a la inversión de la SDP](https://www.sdp.gov.co/gestion-a-la-inversion/programacion-y-seguimiento-a-la-inversion/seguimiento).

La celda de instalación es opcional. Utilizo las dependencias públicas del proyecto y mantengo la documentación del procedimiento en estos notebooks.


---

## 2. Configuración del alcance y del corte

En el siguiente bloque importo las bibliotecas y declaro el plan que voy a consultar. Mantengo el nombre, la sigla y el año inicial como parámetros visibles porque cambiar de plan es una decisión de alcance, no una deducción a partir del último archivo disponible.

Con `CORTE_DESCARGA=None` busco el último corte compatible. Si escribo una fecha ISO concreta, exijo ese corte: no lo sustituyo silenciosamente por otro. Con `INCLUIR_NARRATIVAS=True` exijo Excel de ambas fuentes y PDF de Gestión-Inversión por entidades. Con `False` selecciono el último corte numérico y no descargo PDF.

No exijo una infografía de Avance del Plan para poder limpiar su Excel. La revisión histórica mostró que ese requisito descartaba cortes numéricos válidos. Para el flujo con narrativas, en el catálogo revisado el último corte completo es junio de 2026; para el flujo numérico es septiembre de 2026. Estos resultados se vuelven a determinar al ejecutar el código.

**Configuración por defecto de esta versión:** `CORTE_DESCARGA=None` e `INCLUIR_NARRATIVAS=False`. Adquiero el último corte numérico común. El historial utilizado para control de mediciones está incorporado en el notebook 03; no se descarga como salida operativa. Si activo narrativas, el criterio exige además el PDF por entidades.


---

## 4. Conexión y control de errores de adquisición

Centralizo las solicitudes en una sesión con tiempos de espera, pausas y reintentos para errores HTTP transitorios. Mantengo activa la verificación de certificados y comunico los errores en lugar de dar por completa una descarga que falló.

La revisión histórica encontró diferencias de disponibilidad y versiones. Por ello separo un corte sin un formato publicado de un fallo de conexión o un archivo ilegible. Una consulta fallida no demuestra que la fuente no exista.


---

## 6. Identificación de formatos y selección del plan

Reconozco Excel y PDF por extensión y, cuando hace falta, por encabezados HTTP. Filtro las publicaciones por el nombre o la sigla del plan configurado. Esta decisión evita combinar el cierre de UNCSAB con los primeros reportes de BCS durante 2024.

Conservo como fuentes diferentes Avance del Plan y Gestión-Inversión. No incorporo automáticamente la clasificación independiente Gestión, porque modificaría la cobertura del proyecto. La identificación del PDF que necesito se realiza por contenido en el bloque siguiente.


---

## 7. Reconocimiento del PDF por entidades

Inspecciono temporalmente las primeras páginas de los PDF candidatos para confirmar plan, corte y reporte por entidades, junto con los rótulos de avances/logros y retrasos/soluciones. No dependo de que el nombre incluya exactamente `entidades`: en el catálogo también encontré nombres con `entidad` y nombres históricos diferentes.

Descarto las vistas por sector o estructura y agrupo copias binarias idénticas mediante SHA-256. Si encuentro versiones diferentes del reporte por entidades, solicito resolver esa diferencia antes de procesar. Esta lectura identifica la fuente; la extracción de sus relatos se realiza en el notebook 04.


---

## 8. Selección del último corte compatible

Consulto años y cortes de forma descendente y compruebo los insumos requeridos por el modo elegido. No admito fechas futuras ni considero completo un corte numérico por la sola presencia de una publicación en el catálogo.

La revisión mostró que no todos los trimestres ofrecen los mismos formatos. Por eso distingo compatibilidad numérica de disponibilidad narrativa. Si falta un insumo de un corte solicitado expresamente, comunico el faltante. Si hay varios Excel candidatos, no elijo una versión por su posición en la página.

La fecha resultante se utiliza para ambas fuentes. En la limpieza, `CORTE=None` también selecciona un corte local común con Excel, para evitar que los notebooks 02 y 03 utilicen periodos distintos por una descarga incompleta.


---

## 10. Ejecución de la adquisición

En este bloque ejecuto las consultas, selecciono el corte y guardo los adjuntos por fecha y fuente en Bronce. La salida muestra lo consultado, lo reconocido y lo descargado o revalidado. Mantengo `descargas` en memoria como resumen de esta ejecución.

Después ejecuto 02 para Avance del Plan, 03 para Gestión-Inversión y, cuando exista un PDF compatible, 04 para narrativas. Las cifras de la revisión histórica que documento a continuación no reemplazan la comprobación de disponibilidad que realiza esta celda al ejecutarse.


---

## 13. Resultado y organización de Capa Bronce

```text
Capa Bronce/
└── AAAA-MM-DD/
    ├── avance_plan/
    │   ├── archivo_original.xlsx
    │   └── archivo_original.pdf
    └── gestion_inversion/
        ├── archivo_original.xlsx
        └── reporte_entidades.pdf
```

Los nombres del esquema son ilustrativos: se conservan los nombres de descarga y sus identificadores estables. Excel y PDF comparten la carpeta de su fuente. Si el portal ofrece varios Excel válidos, se conservan según las reglas de selección; no se fuerza un límite artificial de dos archivos.

Bronce representa la conservación de las fuentes originales para su uso posterior en el proyecto de grado. Las fechas de carpeta corresponden al corte seleccionado, no al día de descarga. Un nuevo corte completo genera sus carpetas sin borrar automáticamente los anteriores.

La interpretación de indicadores, unidades, porcentajes, relatos y calidad de datos corresponde a etapas posteriores. Este notebook no asegura exhaustividad del portal, equivalencia literal entre distintas vistas de los reportes ni validez analítica de los valores publicados; implementa la adquisición bajo las reglas descritas.

---

## 11. Qué encontré al revisar todos los cortes publicados desde 2024

Consulté las dos clasificaciones y todos sus periodos para 2024, 2025 y 2026. El catálogo devolvió 21 publicaciones en el ámbito revisado. Leí los 20 Excel enlazados: 17 corresponden a BCS y tres a UNCSAB. Además revisé los cuatro PDF por entidades publicados para BCS. No presupuse que un archivo con corte trimestral tuviera la misma estructura o unidad de observación que el del trimestre siguiente.

| Corte | Plan | Avance del Plan | Gestión-Inversión | Alcance que pude comprobar |
|---|---|---|---|---|
| 2024-03-31 | UNCSAB | No listado en esta clasificación/año | [Excel](https://www.sdp.gov.co/sites/default/files/04a_planaccioncompgestioninversionxentidad20240331.xlsx) | Revisión de estructura; unidad diferente a BCS |
| 2024-05-31 | UNCSAB | [Excel](https://www.sdp.gov.co/sites/default/files/02_avancefisico_20240531.xlsx) | [Excel](https://www.sdp.gov.co/sites/default/files/04a_planaccioncompgestioninversionxentidad20240531.xlsx) | Cierre de plan; no mezclar con actividades BCS |
| 2024-09-30 | BCS | [Excel](https://www.sdp.gov.co/sites/default/files/avancefisicobcs20240930.xlsx) | [Solo PDF](https://www.sdp.gov.co/sites/default/files/componentegestioninversionbcs20240930.pdf) | Limpieza de Avance; PDF con discrepancia de cobertura |
| 2024-12-31 | BCS | [Excel](https://www.sdp.gov.co/sites/default/files/02_avancefisico20241231_act20250219.xlsx) | [Excel](https://www.sdp.gov.co/sites/default/files/pa_cg_ci_20241231_0.xlsx) | Dos Excel compatibles tras adaptar esquemas; PDF con discrepancia |
| 2025-03-31 | BCS | [Excel](https://www.sdp.gov.co/sites/default/files/01_avancefisico20250331.xlsx) | [Excel](https://www.sdp.gov.co/sites/default/files/pa_cg_ci_20250331_act20250519.xlsx) | Numérico; cinco registros con fecha fuente de 2024 |
| 2025-06-30 | BCS | [Excel](https://www.sdp.gov.co/sites/default/files/01_avancefisico20250630.xlsx) | [Excel](https://www.sdp.gov.co/sites/default/files/pa_cg_ci_20250630_ver20250730.xlsx) | Numérico; Avance todavía en escala 0–100 |
| 2025-09-30 | BCS | [Excel](https://www.sdp.gov.co/sites/default/files/01_avancefisico20250930.xlsx) | [Excel](https://www.sdp.gov.co/sites/default/files/planacion_gestion-inversion_20250930_entidad_v20251120.xlsx) | Numérico; Avance pasa a fracciones |
| 2025-12-31 | BCS | [Excel](https://www.sdp.gov.co/sites/default/files/01_avancefisico20251231_0_0.xlsx) | [Excel](https://www.sdp.gov.co/sites/default/files/planaciongestioninventidad20251231_v20260415.xlsx) | Numérico; versión revisada del Excel |
| 2026-03-31 | BCS | [Excel](https://www.sdp.gov.co/sites/default/files/01_avancefisico20260331.xlsx) | [Excel](https://www.sdp.gov.co/sites/default/files/planacciongestioninversionentidad20260331_ver20260505_0.xlsx) | Numérico y narrativo; diccionario omite FE_DE_ERRATAS |
| 2026-06-30 | BCS | [Excel](https://www.sdp.gov.co/sites/default/files/01_avancefisico20260630.xlsx) | [Excel](https://www.sdp.gov.co/sites/default/files/planacciongestioninversion20260630_ver20260728.xlsx) | Numérico y narrativo; corte de los resultados locales |
| 2026-09-30 | BCS | [Excel](https://www.sdp.gov.co/sites/default/files/avanceplan_30092026.xlsx) | [Excel](https://www.sdp.gov.co/sites/default/files/planacciongestioninversion20260930_ver20261027.xlsx) | Numérico; nuevo formato de pivotes; sin PDF de Gestión publicado |

En marzo y mayo de 2024, Gestión-Inversión presenta `DatosMetasSectoriales`, `DatosIndicadores` y `DatosProyectosMetas`. En la última tabla encontré 2.078 y 2.079 filas, respectivamente. No la equiparé con una actividad de BCS ni trasladé sus porcentajes a ese plan. La ausencia de una tabla moderna de actividades en ese esquema es una diferencia de fuente, no una invitación a fabricar equivalencias.

Para reproducir un corte numérico publicado puedo fijar `CORTE_DESCARGA` y desactivar `INCLUIR_NARRATIVAS`; después selecciono el mismo corte en 02 y 03. En un corte sin Excel de Gestión-Inversión, como septiembre de 2024, no genero una base tabular de actividades a partir de un formato que no se publicó. Las revisiones y pruebas históricas se realizaron fuera de la carpeta de entrega; sus conclusiones y fuentes quedan documentadas en estos notebooks.


</details>
